# 1. Understand and prepare the data

**Question:** What information do we have, and how can we test on customers the model has not seen?

Start here. Select the project Python kernel and run cells from top to bottom with Shift+Enter.
Markdown cells explain the next code cell. A DataFrame is a table; a column is one variable.
`churn = 1` means the customer left; `0` means they stayed.

This is supervised machine learning: examples include the outcome we want to predict.
It does not use a language model or generate text.

In [ ]:
import sys
from pathlib import Path

# Jupyter may start in the project root or in notebooks/.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data" / "source.json").exists():
    raise FileNotFoundError("Open Jupyter from the RetainAI project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / "output"
OUTPUT.mkdir(exist_ok=True)

## Read the original CSV

The original data is kept in `data/`. We check its SHA-256 fingerprint before reading it.
This detects accidental edits; do not change the recorded fingerprint to silence a failure.
Column names are simplified in memory, and display IDs are added. These are not real customer IDs.

In [ ]:
import hashlib
import json

import pandas as pd

from retainai.features import INPUT_COLUMNS, validate_customers

source = json.loads((ROOT / "data/source.json").read_text(encoding="utf-8"))
raw_path = ROOT / "data/customer_churn.csv"
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == source["sha256"], (
    "Restore the original source CSV."
)
customers = pd.read_csv(raw_path)
customers.columns = ["_".join(name.lower().split()) for name in customers.columns]
customers.insert(0, "customer_id", [f"CUST_{i + 1:05d}" for i in range(len(customers))])
validate_customers(
    customers
)  # Reject missing, nonfinite, negative, or invalid coded inputs.
assert customers["churn"].isin([0, 1]).all() and customers["churn"].nunique() == 2
customers.head()

## Inspect before modelling

`shape` gives rows and columns, `isna` counts missing cells, and `mean` of a 0/1 target is the churn fraction.
Exact duplicates compare every original column. Repeated predictor profiles compare only the 12 inputs
the model will see. Those are different counts. We keep all rows because the source does not tell us
whether identical anonymous records are different people or duplicated entries.

UCI describes charge tiers as 0–9, but the original CSV also contains seven rows coded 10.
We preserve those rows and explicitly allow 0–10 rather than guessing a correction.

In [ ]:
summary = {
    "rows": len(customers),
    "missing_cells": int(customers.isna().sum().sum()),
    "exact_duplicate_rows": int(
        customers.drop(columns="customer_id").duplicated().sum()
    ),
    "repeated_predictor_rows": int(customers[INPUT_COLUMNS].duplicated().sum()),
    "unique_predictor_profiles": int(len(customers[INPUT_COLUMNS].drop_duplicates())),
    "churn_rate": float(customers["churn"].mean()),
}
display(pd.Series(summary))
display(customers.describe().round(2))
customers["churn"].value_counts().rename(index={0: "stayed", 1: "left"}).plot.bar(
    title="Customers by observed outcome", ylabel="Number of customers", rot=0
)

## Split into training, validation, and test tables

Training teaches the models. Validation chooses between models. Test measures the selected model
at the end. We aim for approximately 70% / 15% / 15% of profiles, so row percentages can differ.

Randomly splitting individual rows could put identical inputs in training and test. Instead,
factorize assigns the same integer group to identical predictor rows; GroupShuffleSplit keeps
each group together. This reduces a specific source of overly optimistic evaluation.
`random_state=42` makes the split repeatable; it is not a quality score.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = pd.factorize(pd.MultiIndex.from_frame(customers[INPUT_COLUMNS]))[0]
first = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, rest_idx = next(first.split(customers, groups=groups))
rest = customers.iloc[rest_idx]
second = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(second.split(rest, groups=groups[rest_idx]))
splits = {
    "train": customers.iloc[train_idx].copy(),
    "validation": rest.iloc[val_idx].copy(),
    "test": rest.iloc[test_idx].copy(),
}
pd.DataFrame(
    {
        name: {"rows": len(part), "churn_rate": part.churn.mean()}
        for name, part in splits.items()
    }
).T

## Check the split and save it

An `assert` is a small check that stops execution when a condition is false. Here it verifies
that no predictor profile crosses partitions, every row appears once, and both outcomes are present.
These checks belong next to the code they protect rather than in a separate training test framework.

Rerunning this stage invalidates later results, so we remove only the known generated downstream files.

In [ ]:
profiles = [
    set(part[INPUT_COLUMNS].itertuples(index=False, name=None))
    for part in splits.values()
]
assert all(
    profiles[i].isdisjoint(profiles[j]) for i in range(3) for j in range(i + 1, 3)
)
combined = pd.concat(splits.values())
assert len(combined) == combined.customer_id.nunique() == source["rows"]
assert all(part.churn.nunique() == 2 for part in splits.values())

for name in ["model.joblib", "metrics.json", "scores.csv", "drift.json"]:
    (OUTPUT / name).unlink(missing_ok=True)
for name, part in splits.items():
    part.to_csv(OUTPUT / f"{name}.csv", index=False)
validate_customers(splits["test"].head(10)).to_csv(
    OUTPUT / "sample_customers.csv", index=False
)
summary["splits"] = {name: len(part) for name, part in splits.items()}
(OUTPUT / "data_report.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
print("Saved split CSVs, an unlabeled example batch, and data_report.json in output/.")

## Before moving on

You should see 3,150 rows, 300 exact duplicates, 314 repeated predictor rows, 2,836 unique profiles,
and split sizes of 2,224 / 458 / 468. Explain why grouping is needed before opening `02_model.ipynb`.
Do not tune the split after looking at test performance.